# Lecture 7: The Density Matrix and Partial Trace
### Computational Companion — Marginalization, Purity, Entropy, Measurement, and No-Signaling

This notebook verifies every claim in Lecture 7:

1. **Partial trace as marginalization** — reduced density matrix from tracing out Bob
2. **Product vs entangled** — rank-1 projector vs maximally mixed
3. **Purity and von Neumann entropy** — scalar entanglement measures
4. **Mixed states** — classical mixture vs partial trace give identical ρ
5. **Measurement as CNOT** — von Neumann model: entangle, then trace
6. **No-signaling** — Bob's operations don't change Alice's ρ
7. **Random state entanglement spectrum** — eigenvalue distributions

**Convention:** ℏ = 1 throughout.

**Reference:** Lecture 7 notes ("Quantum Systems via Linear Algebra")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import expm, logm

# ── Pauli matrices ──
I2 = np.eye(2, dtype=complex)
Z = np.array([[1, 0], [0, -1]], dtype=complex)
X = np.array([[0, 1], [1, 0]], dtype=complex)
Y = np.array([[0, -1j], [1j, 0]], dtype=complex)

# ── Basis vectors ──
e0 = np.array([1, 0], dtype=complex)
e1 = np.array([0, 1], dtype=complex)

def kron(a, b):
    return np.kron(a, b)

def partial_trace_B(rho_AB, dA=2, dB=2):
    """Partial trace over subsystem B of a dA*dB × dA*dB density matrix."""
    rho_A = np.zeros((dA, dA), dtype=complex)
    for i in range(dA):
        for j in range(dA):
            for k in range(dB):
                rho_A[i, j] += rho_AB[i*dB + k, j*dB + k]
    return rho_A

def partial_trace_A(rho_AB, dA=2, dB=2):
    """Partial trace over subsystem A of a dA*dB × dA*dB density matrix."""
    rho_B = np.zeros((dB, dB), dtype=complex)
    for i in range(dB):
        for j in range(dB):
            for k in range(dA):
                rho_B[i, j] += rho_AB[k*dB + i, k*dB + j]
    return rho_B

def expect(M, v):
    return np.real(v.conj() @ M @ v)

def purity(rho):
    return np.real(np.trace(rho @ rho))

def von_neumann_entropy(rho):
    evals = np.linalg.eigvalsh(rho)
    evals = evals[evals > 1e-15]  # avoid log(0)
    return -np.sum(evals * np.log2(evals))

print("Setup complete: Pauli matrices, partial trace, purity, von Neumann entropy.")

## 1. Partial Trace — Reduced Density Matrix

Given a composite state $|\mathbf{v}\rangle \in \mathbb{C}^2 \otimes \mathbb{C}^2$, Alice's reduced density matrix is
$$\rho^{(A)} = \text{Tr}_B(|\mathbf{v}\rangle\langle\mathbf{v}|)$$

We compute this for product states and entangled states, verifying all properties.

In [ ]:
# ── 1. Partial Trace ──

print("=" * 65)
print("PARTIAL TRACE: reduced density matrix")
print("=" * 65)

# ── Product state: e0 ⊗ x+ ──
x_plus = (e0 + e1) / np.sqrt(2)
v_product = kron(e0, x_plus)
rho_AB_prod = np.outer(v_product, v_product.conj())
rho_A_prod = partial_trace_B(rho_AB_prod)

print("\nProduct state: e0 ⊗ x+")
print(f"ρ^(A) =\n{np.round(rho_A_prod, 4)}")
print(f"Expected: |e0⟩⟨e0| = diag(1, 0)")
print(f"Match: {np.allclose(rho_A_prod, np.outer(e0, e0.conj()))}")
print(f"Eigenvalues: {np.round(np.linalg.eigvalsh(rho_A_prod), 4)}")
print(f"Tr(ρ) = {np.trace(rho_A_prod).real:.4f}")
print(f"ρ² = ρ? {np.allclose(rho_A_prod @ rho_A_prod, rho_A_prod)}")

# ── Singlet ──
v_singlet = (kron(e0, e1) - kron(e1, e0)) / np.sqrt(2)
rho_AB_sing = np.outer(v_singlet, v_singlet.conj())
rho_A_sing = partial_trace_B(rho_AB_sing)

print("\nSinglet state: (e0⊗f1 − e1⊗f0)/√2")
print(f"ρ^(A) =\n{np.round(rho_A_sing, 4)}")
print(f"Expected: ½I")
print(f"Match: {np.allclose(rho_A_sing, 0.5 * I2)}")
print(f"Eigenvalues: {np.round(np.linalg.eigvalsh(rho_A_sing), 4)}")
print(f"Tr(ρ) = {np.trace(rho_A_sing).real:.4f}")
print(f"ρ² = ρ? {np.allclose(rho_A_sing @ rho_A_sing, rho_A_sing)}  (mixed, NOT a projector)")

# ── Properties check ──
print("\n" + "=" * 65)
print("PROPERTIES of ρ^(A)")
print("=" * 65)
for name, rho in [("Product", rho_A_prod), ("Singlet", rho_A_sing)]:
    print(f"\n  {name}:")
    print(f"    Self-adjoint: {np.allclose(rho, rho.conj().T)}")
    print(f"    Tr(ρ) = {np.trace(rho).real:.6f}")
    print(f"    PSD (min eigenvalue ≥ 0): {np.linalg.eigvalsh(rho).min():.6f}")

## 2. Expectation Values via Density Matrix

For any Alice observable $L$: $\langle L \rangle = \text{Tr}(L\, \rho^{(A)})$.

We verify this matches the direct computation $\langle \mathbf{v} | (L \otimes I) | \mathbf{v} \rangle$.

In [ ]:
# ── 2. Expectation Values via Density Matrix ──

print("=" * 65)
print("EXPECTATION VALUES: Tr(Lρ) vs ⟨v|(L⊗I)|v⟩")
print("=" * 65)

test_states = [
    ("Product: e0 ⊗ x+", v_product),
    ("Singlet", v_singlet),
    ("Bell |Φ+⟩", (kron(e0, e0) + kron(e1, e1)) / np.sqrt(2)),
]

for name, v in test_states:
    rho_AB = np.outer(v, v.conj())
    rho_A = partial_trace_B(rho_AB)
    print(f"\n  {name}:")
    for obs_name, L in [("X", X), ("Y", Y), ("Z", Z)]:
        # Method 1: density matrix
        val_dm = np.real(np.trace(L @ rho_A))
        # Method 2: direct
        val_direct = expect(kron(L, I2), v)
        print(f"    ⟨{obs_name}⟩: Tr(Lρ) = {val_dm:.6f}, ⟨v|(L⊗I)|v⟩ = {val_direct:.6f}, match = {np.isclose(val_dm, val_direct)}")

## 3. Purity and Von Neumann Entropy

- **Purity**: $\gamma = \text{Tr}(\rho^2) = \sum_i \sigma_i^4$. Ranges from $1/n$ (maximally entangled) to $1$ (product).
- **Von Neumann entropy**: $S = -\text{Tr}(\rho \log_2 \rho) = -\sum_i \lambda_i \log_2 \lambda_i$. Ranges from $0$ (product) to $\log_2 n$ (maximally entangled).

In [ ]:
# ── 3. Purity and Von Neumann Entropy ──

print("=" * 65)
print("PURITY and VON NEUMANN ENTROPY")
print("=" * 65)

states = [
    ("Product: e0⊗x+", v_product),
    ("Singlet", v_singlet),
    ("Bell |Φ+⟩", (kron(e0, e0) + kron(e1, e1)) / np.sqrt(2)),
    ("Partially entangled", np.sqrt(0.8)*kron(e0,e0) + np.sqrt(0.2)*kron(e1,e1)),
    ("Partially entangled (0.9/0.1)", np.sqrt(0.9)*kron(e0,e0) + np.sqrt(0.1)*kron(e1,e1)),
]

print(f"\n  {'State':<35} {'Purity':>8} {'Entropy':>9} {'Eigenvalues'}")
print("  " + "-" * 75)
for name, v in states:
    rho_AB = np.outer(v, v.conj())
    rho_A = partial_trace_B(rho_AB)
    g = purity(rho_A)
    S = von_neumann_entropy(rho_A)
    evals = np.sort(np.linalg.eigvalsh(rho_A))[::-1]
    print(f"  {name:<35} {g:8.4f} {S:9.4f}   {np.round(evals, 4)}")

# Also verify SVD connection: eigenvalues of ρ_A = σ_i² of coefficient matrix
print("\n" + "=" * 65)
print("SVD CONNECTION: eigenvalues of ρ^(A) = σᵢ² of coefficient matrix")
print("=" * 65)
for name, v in states:
    Psi = v.reshape(2, 2)
    _, sigmas, _ = np.linalg.svd(Psi)
    rho_A = partial_trace_B(np.outer(v, v.conj()))
    evals_rho = np.sort(np.linalg.eigvalsh(rho_A))[::-1]
    print(f"  {name:<35}  σ² = {np.round(sigmas**2, 4)},  evals(ρ) = {np.round(evals_rho, 4)},  match = {np.allclose(np.sort(sigmas**2)[::-1], evals_rho)}")

## 4. Mixed States — Classical Mixture vs Partial Trace

A density matrix $\rho = \sum_i p_i |\mathbf{v}_i\rangle\langle\mathbf{v}_i|$ (classical mixture) is operationally indistinguishable from the partial trace of some pure entangled state. We verify: construct $\rho = \frac{1}{2}I$ two ways — as a classical mixture and as a partial trace — and confirm they're identical.

In [ ]:
# ── 4. Mixed States ──

print("=" * 65)
print("MIXED STATES: classical mixture vs partial trace")
print("=" * 65)

# Method 1: Classical mixture — 50% |e0⟩, 50% |e1⟩
rho_mix1 = 0.5 * np.outer(e0, e0.conj()) + 0.5 * np.outer(e1, e1.conj())
print(f"\nMethod 1: Classical mixture ½|e0⟩⟨e0| + ½|e1⟩⟨e1|:")
print(f"{np.round(rho_mix1, 4)}")

# Method 2: Classical mixture — 50% |x+⟩, 50% |x−⟩
x_minus = (e0 - e1) / np.sqrt(2)
rho_mix2 = 0.5 * np.outer(x_plus, x_plus.conj()) + 0.5 * np.outer(x_minus, x_minus.conj())
print(f"\nMethod 2: Classical mixture ½|x+⟩⟨x+| + ½|x−⟩⟨x−|:")
print(f"{np.round(rho_mix2, 4)}")

# Method 3: Partial trace of singlet
rho_pt = rho_A_sing
print(f"\nMethod 3: Partial trace of singlet:")
print(f"{np.round(rho_pt, 4)}")

# Method 4: Partial trace of Bell |Φ+⟩
v_bell = (kron(e0, e0) + kron(e1, e1)) / np.sqrt(2)
rho_bell_A = partial_trace_B(np.outer(v_bell, v_bell.conj()))
print(f"\nMethod 4: Partial trace of Bell |Φ+⟩:")
print(f"{np.round(rho_bell_A, 4)}")

print(f"\nAll four give ½I:")
print(f"  Mix1 = Mix2: {np.allclose(rho_mix1, rho_mix2)}")
print(f"  Mix1 = PT(singlet): {np.allclose(rho_mix1, rho_pt)}")
print(f"  Mix1 = PT(Bell): {np.allclose(rho_mix1, rho_bell_A)}")
print(f"\n→ Classical uncertainty and quantum entanglement are operationally indistinguishable!")
print(f"  This is the purification theorem: every mixed state arises as a partial trace.")

## 5. Measurement as CNOT — Von Neumann Model

The measurement interaction is a CNOT gate: system controls whether apparatus flips.
$$U(\alpha|e_0\rangle + \beta|e_1\rangle) \otimes |f_0\rangle = \alpha|e_0\rangle|f_0\rangle + \beta|e_1\rangle|f_1\rangle$$

After tracing out the apparatus, the system's density matrix shows the "collapse."

In [ ]:
# ── 5. Measurement as CNOT ──

print("=" * 65)
print("VON NEUMANN MEASUREMENT: CNOT + partial trace = collapse")
print("=" * 65)

# CNOT gate (system = control, apparatus = target)
CNOT = np.array([
    [1, 0, 0, 0],
    [0, 1, 0, 0],
    [0, 0, 0, 1],
    [0, 0, 1, 0]
], dtype=complex)
print(f"CNOT gate:\n{CNOT.real.astype(int)}")
print(f"Unitary: {np.allclose(CNOT @ CNOT.conj().T, np.eye(4))}")

# System in superposition
alpha = np.cos(np.pi/5)
beta = np.exp(1j * np.pi/3) * np.sin(np.pi/5)
psi_sys = alpha * e0 + beta * e1
f0 = e0  # apparatus ready state

# Pre-measurement: |ψ⟩ ⊗ |f0⟩
v_pre = kron(psi_sys, f0)
rho_pre = np.outer(v_pre, v_pre.conj())
rho_sys_pre = partial_trace_B(rho_pre)

print(f"\nPre-measurement system ρ:")
print(f"{np.round(rho_sys_pre, 4)}")
print(f"Purity = {purity(rho_sys_pre):.4f} (pure state)")

# Apply CNOT
v_post = CNOT @ v_pre
rho_post = np.outer(v_post, v_post.conj())

# System's density matrix after CNOT (trace out apparatus)
rho_sys_post = partial_trace_B(rho_post)
print(f"\nPost-CNOT system ρ (trace out apparatus):")
print(f"{np.round(rho_sys_post, 4)}")
print(f"Purity = {purity(rho_sys_post):.4f} (mixed! < 1)")

# This should be diagonal with |α|² and |β|²
expected_rho = np.diag([abs(alpha)**2, abs(beta)**2]).astype(complex)
print(f"\nExpected (diagonal with |α|², |β|²):")
print(f"{np.round(expected_rho, 4)}")
print(f"Match: {np.allclose(rho_sys_post, expected_rho)}")

print(f"\n→ The off-diagonal coherences are gone!")
print(f"  Pre: ρ₀₁ = {rho_sys_pre[0,1]:.4f} (coherent superposition)")
print(f"  Post: ρ₀₁ = {rho_sys_post[0,1]:.4f} (decoherence from entanglement)")
print(f"\n→ 'Collapse' = entangle with apparatus + trace out apparatus.")

## 6. No-Signaling Theorem

Bob's operations (unitaries or measurements) cannot change Alice's reduced density matrix $\rho^{(A)}$. We verify numerically for several Bob operations on the singlet and other entangled states.

In [ ]:
# ── 6. No-Signaling ──

print("=" * 65)
print("NO-SIGNALING: Bob's operations don't change Alice's ρ")
print("=" * 65)

# Test on singlet
rho_AB_sing = np.outer(v_singlet, v_singlet.conj())
rho_A_original = partial_trace_B(rho_AB_sing)
print(f"\nOriginal Alice ρ (singlet):")
print(f"{np.round(rho_A_original, 4)}")

# Bob's unitary operations
bob_unitaries = [
    ("I (do nothing)", I2),
    ("X (bit flip)", X),
    ("Z (phase flip)", Z),
    ("Y", Y),
    ("Hadamard", np.array([[1,1],[1,-1]], dtype=complex)/np.sqrt(2)),
    ("Random U", expm(1j * (0.3*X + 0.7*Y + 0.5*Z))),
]

print("\nBob applies unitary UB (acts as I⊗UB):")
for name, UB in bob_unitaries:
    IUB = kron(I2, UB)
    rho_AB_new = IUB @ rho_AB_sing @ IUB.conj().T
    rho_A_new = partial_trace_B(rho_AB_new)
    changed = not np.allclose(rho_A_new, rho_A_original)
    print(f"  {name:<20} → Alice ρ changed? {changed}  (max diff = {np.max(np.abs(rho_A_new - rho_A_original)):.2e})")

# Bob's measurement (projective) — averaged over outcomes
print("\nBob measures observable M (averaged over outcomes):")
bob_measurements = [
    ("Z measurement", [np.outer(e0, e0.conj()), np.outer(e1, e1.conj())]),
    ("X measurement", [np.outer(x_plus, x_plus.conj()), np.outer((e0-e1)/np.sqrt(2), ((e0-e1)/np.sqrt(2)).conj())]),
]

for name, projectors in bob_measurements:
    rho_A_avg = np.zeros((2, 2), dtype=complex)
    for Pk in projectors:
        IPk = kron(I2, Pk)
        rho_A_avg += partial_trace_B(IPk @ rho_AB_sing @ IPk)
    changed = not np.allclose(rho_A_avg, rho_A_original)
    print(f"  {name:<20} → Alice ρ changed? {changed}  (max diff = {np.max(np.abs(rho_A_avg - rho_A_original)):.2e})")

# Test on a non-singlet entangled state too
print("\n" + "=" * 65)
print("NO-SIGNALING on Bell |Φ+⟩ = (|00⟩+|11⟩)/√2")
print("=" * 65)
rho_AB_bell = np.outer(v_bell, v_bell.conj())
rho_A_bell = partial_trace_B(rho_AB_bell)
for name, UB in bob_unitaries:
    IUB = kron(I2, UB)
    rho_A_new = partial_trace_B(IUB @ rho_AB_bell @ IUB.conj().T)
    print(f"  {name:<20} → changed? {not np.allclose(rho_A_new, rho_A_bell)}  (max diff = {np.max(np.abs(rho_A_new - rho_A_bell)):.2e})")

## 7. Random State Entanglement Spectrum

For Haar-random pure states of a 2-qubit system, we compute the eigenvalue spectrum of $\rho^{(A)}$, the purity, and the von Neumann entropy. This shows the typical entanglement structure of random states.

In [ ]:
# ── 7. Random State Entanglement Spectrum ──

print("=" * 65)
print("RANDOM STATE SPECTRUM: purity and entropy distributions")
print("=" * 65)

rng = np.random.default_rng(42)
n_samples = 10000
purities = np.zeros(n_samples)
entropies = np.zeros(n_samples)
max_evals = np.zeros(n_samples)

for i in range(n_samples):
    v = rng.standard_normal(4) + 1j * rng.standard_normal(4)
    v /= np.linalg.norm(v)
    rho_AB = np.outer(v, v.conj())
    rho_A = partial_trace_B(rho_AB)
    purities[i] = purity(rho_A)
    entropies[i] = von_neumann_entropy(rho_A)
    max_evals[i] = np.max(np.linalg.eigvalsh(rho_A))

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(purities, bins=80, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].axvline(1.0, color='red', ls='--', lw=2, label='Product (γ=1)')
axes[0].axvline(0.5, color='green', ls='--', lw=2, label='Max entangled (γ=½)')
axes[0].set_xlabel('Purity Tr(ρ²)'); axes[0].set_ylabel('Count')
axes[0].set_title('Purity distribution (10k Haar-random states)')
axes[0].legend(fontsize=9); axes[0].grid(alpha=0.3)

axes[1].hist(entropies, bins=80, color='darkorange', edgecolor='white', alpha=0.8)
axes[1].axvline(0, color='red', ls='--', lw=2, label='Product (S=0)')
axes[1].axvline(1.0, color='green', ls='--', lw=2, label='Max entangled (S=1 bit)')
axes[1].set_xlabel('Von Neumann entropy S (bits)'); axes[1].set_ylabel('Count')
axes[1].set_title('Entropy distribution')
axes[1].legend(fontsize=9); axes[1].grid(alpha=0.3)

axes[2].scatter(purities, entropies, alpha=0.1, s=3, c='steelblue')
# Theoretical curve: λ, 1-λ with purity = λ² + (1-λ)²
lams = np.linspace(0.5, 1.0, 200)
p_theory = lams**2 + (1-lams)**2
s_theory = np.array([-l*np.log2(l) - (1-l)*np.log2(1-l) if l < 1 else 0 for l in lams])
axes[2].plot(p_theory, s_theory, 'r-', lw=2, label='Theoretical curve')
axes[2].set_xlabel('Purity'); axes[2].set_ylabel('Entropy (bits)')
axes[2].set_title('Purity vs Entropy')
axes[2].legend(); axes[2].grid(alpha=0.3)

plt.suptitle('Entanglement properties of Haar-random 2-qubit states', fontsize=13)
plt.tight_layout(); plt.show()

print(f"Mean purity: {purities.mean():.4f} (theory ≈ 0.533)")
print(f"Mean entropy: {entropies.mean():.4f} bits")
print(f"Fraction with purity > 0.99 (near-product): {np.sum(purities > 0.99)/n_samples:.4f}")

## 8. Classical-Quantum Analogy Table — Verified

We verify the analogy from the lecture: marginalization of a classical joint distribution parallels partial trace of a quantum state. For a product state, the marginal is independent (just like classical independence). For an entangled state, the marginal is "correlated" (mixed).

In [ ]:
# ── 8. Classical-Quantum Analogy ──

print("=" * 65)
print("CLASSICAL-QUANTUM ANALOGY: marginalization")
print("=" * 65)

# Classical: joint distribution p(a,b) on {0,1} × {0,1}
# Independent: p(a,b) = p_A(a) * p_B(b)
p_A = np.array([0.7, 0.3])
p_B = np.array([0.4, 0.6])
p_independent = np.outer(p_A, p_B)
print(f"\nClassical independent joint:")
print(f"  p(a,b) = {np.round(p_independent, 3)}")
print(f"  Marginal p(a) = {np.round(p_independent.sum(axis=1), 3)} (= p_A ✓)")

# Classical: correlated joint
p_correlated = np.array([[0.5, 0.0], [0.0, 0.5]])
print(f"\nClassical perfectly correlated:")
print(f"  p(a,b) = {p_correlated}")
print(f"  Marginal p(a) = {p_correlated.sum(axis=1)} (uniform — no info about a alone!)")

# Quantum analog
print("\n" + "-" * 40)
print("Quantum analogs:")

# Product state
psi_A = np.sqrt(0.7) * e0 + np.sqrt(0.3) * e1
psi_B = np.sqrt(0.4) * e0 + np.sqrt(0.6) * e1
v_prod = kron(psi_A, psi_B)
rho_A_prod = partial_trace_B(np.outer(v_prod, v_prod.conj()))
print(f"\nProduct state:")
print(f"  ρ^(A) eigenvalues = {np.round(np.sort(np.linalg.eigvalsh(rho_A_prod))[::-1], 4)}")
print(f"  Purity = {purity(rho_A_prod):.4f} (pure — Alice has full info)")

# Singlet (maximally entangled — like perfectly correlated classical)
print(f"\nSinglet (maximally entangled):")
print(f"  ρ^(A) eigenvalues = {np.round(np.sort(np.linalg.eigvalsh(rho_A_sing))[::-1], 4)}")
print(f"  Purity = {purity(rho_A_sing):.4f} (maximally mixed — Alice has NO info)")

print(f"\n→ Classical marginalization ↔ quantum partial trace")
print(f"  Classical independence ↔ product state (pure marginal)")
print(f"  Classical correlation ↔ entanglement (mixed marginal)")